# TRUSTDESC capstone Phase 1 handover

This notebook runs the portable **bounded reimplementation**. It does not reproduce the original 208-task results. Read `docs/COLAB_HANDOVER.md` and `docs/REPRODUCTION_STATUS.md` in the archive.

Track A uses CPU, synthetic tools and replay responses (no inference). Track B is an optional, unrun Qwen 1.5B GPU inference experiment, a different model setting. No Drive mount, ChatGPT connection, API key or original paper upload is required. Do not execute unknown MCP tools here.


In [ ]:
import sys, platform, shutil, subprocess
from pathlib import Path
assert sys.version_info >= (3, 11), "Use Python 3.11 or newer"
print({"python": sys.version, "platform": platform.platform(), "disk": shutil.disk_usage(".").free})
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)


## Upload and verify the project archive

Upload only the delivered `trustdesc-phase1.zip`. Copy its SHA-256 from the companion `.sha256` file into `EXPECTED_SHA256` below before running. This is a manual upload to the Colab session you selected; Codex does not connect to your account. Extraction verifies member hashes before any project code is executed.


In [ ]:
from google.colab import files
import hashlib, json, zipfile
from pathlib import Path, PurePosixPath
EXPECTED_SHA256 = "PASTE_THE_DELIVERED_64_CHARACTER_SHA256_HERE"
assert len(EXPECTED_SHA256) == 64 and all(c in "0123456789abcdef" for c in EXPECTED_SHA256), "Paste the delivered checksum first"
uploaded = files.upload()
assert len(uploaded) == 1, "Upload only the project ZIP"
name, archive_bytes = next(iter(uploaded.items()))
assert name.endswith(".zip")
assert hashlib.sha256(archive_bytes).hexdigest() == EXPECTED_SHA256, "Archive checksum mismatch"
archive_path = Path("/content/trustdesc-phase1.zip")
archive_path.write_bytes(archive_bytes)
ROOT = Path("/content/trustdesc-reproduction")
assert not ROOT.exists(), "Use a fresh runtime or a reviewed fresh extraction location"
with zipfile.ZipFile(archive_path) as z:
    infos = z.infolist()
    assert len(infos) <= 10000 and sum(i.file_size for i in infos) < 100_000_000
    assert len(z.namelist()) == len(set(z.namelist()))
    for i in infos:
        p = PurePosixPath(i.filename)
        assert not p.is_absolute() and ".." not in p.parts and "\\" not in i.filename
        assert p.parts[0] == "trustdesc-reproduction"
        assert ((i.external_attr >> 16) & 0o170000) != 0o120000
    manifest = json.loads(z.read("trustdesc-reproduction/PORTABLE_MANIFEST.json"))
    expected = {"trustdesc-reproduction/" + n for n in manifest["files"]} | {"trustdesc-reproduction/PORTABLE_MANIFEST.json"}
    assert set(z.namelist()) == expected
    for n, sha in manifest["files"].items():
        assert hashlib.sha256(z.read("trustdesc-reproduction/" + n)).hexdigest() == sha
    z.extractall("/content")
print({"verified_project_files": len(manifest["files"]), "source_commit": manifest["git_commit"]})


## Track A: CPU workflow

All following core steps need only the Python standard library. No `pip install` is necessary. Each run gets a new output directory. Local validation executes only cells tagged `portable`; upload, GPU and export cells remain unexecuted.


In [ ]:
import os, sys, json, subprocess
from datetime import datetime, timezone
from pathlib import Path
ROOT = Path(ROOT).resolve()
assert (ROOT / "src/trustdesc").is_dir()
RUN_ROOT = Path(globals().get("RUN_ROOT", ROOT / "experiments/runs" / datetime.now(timezone.utc).strftime("colab-%Y%m%dT%H%M%S%fZ")))
RUN_ROOT.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "PYTHONPATH": str(ROOT / "src")}
RUN_GPU_EXPERIMENT = False  # Explicit opt-in required for optional downloads/inference.
print({"project": str(ROOT), "run_directory": str(RUN_ROOT), "replay_is_inference": False})


In [ ]:
subprocess.run([sys.executable, "scripts/run_tests.py"], cwd=ROOT, env=ENV, check=True)
test_results = json.loads((ROOT / "reports/test_results.json").read_text())
assert test_results["success"] and not test_results["failures"] and not test_results["errors"] and not test_results["skipped"]
print(test_results)


In [ ]:
subprocess.run([sys.executable, "scripts/prepare_dataset.py", "tests/fixtures/tasks.jsonl", "--output", str(RUN_ROOT / "prepared")], cwd=ROOT, env=ENV, check=True)
data_manifest = json.loads((RUN_ROOT / "prepared/manifest.json").read_text())
assert data_manifest["stats"]["records"] == 8
assert data_manifest["stats"]["origin_counts"] == {"synthetic": 8}
assert data_manifest["stats"]["missing_labels"] == 8
print(data_manifest["stats"])


In [ ]:
subprocess.run([sys.executable, "-m", "trustdesc", "demo", "--output", str(RUN_ROOT / "fixture")], cwd=ROOT, env=ENV, check=True)
verification = json.loads((RUN_ROOT / "fixture/verification.json").read_text())
decisions = {r["claim_id"]: r["verdict"] for r in verification["decisions"]}
assert decisions["auto_prefix"] == "refuted" and decisions["internal"] == "not_runtime_verifiable"
assert verification["trust_decision"] == "not_assessed"
print({"synthetic_decisions": decisions, "description": verification["full"], "paper_reproduced": False})


In [ ]:
subprocess.run([sys.executable, "scripts/run_local_experiments.py"], cwd=ROOT, env=ENV, check=True)
engineering = json.loads((ROOT / "reports/local_experiments.json").read_text())
assert engineering["paper_metrics"] is None
print(engineering["summary"])


## Track B: optional GPU inference — different model, not original reproduction

Leave `RUN_GPU_EXPERIMENT=False` for CPU work. To try real inference in your own GPU runtime, set it to `True` in the setup cell, inspect available GPU/RAM/disk, then run the optional cells. This opts into public package/model downloads. No credentials, private source uploads or paid API calls are used. The GPU environment and model inference have **not been validated in this Mac session**. A small model may fail strict output checks; preserve those failures.

After package installation Colab may request a runtime restart. Rerun setup and point ROOT at the already verified extraction. Never rerun extraction over an existing project. Preserve package versions and raw errors. The source model revision prefix is fixed in `configs/small_model.json`; full resolved commit is logged on load.


In [ ]:
if RUN_GPU_EXPERIMENT:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "notebooks/colab/requirements-gpu.txt"], cwd=ROOT, check=True)
    # If Colab asks, restart now, then rerun setup/import cells (not extraction).
else:
    print("GPU package/model downloads are disabled.")


In [ ]:
if RUN_GPU_EXPERIMENT:
    import torch
    assert torch.cuda.is_available(), "Select a CUDA GPU runtime"
    print({"torch": torch.__version__, "GPU": torch.cuda.get_device_name(0), "memory": torch.cuda.mem_get_info()})
    (RUN_ROOT / "pip-freeze.txt").write_text(subprocess.run([sys.executable, "-m", "pip", "freeze"], text=True, capture_output=True, check=True).stdout)
    gpu_out = RUN_ROOT / ("qwen-desc-" + datetime.now(timezone.utc).strftime("%H%M%S%f"))
    # Failure is preserved with nonzero return code; do not turn it into a success label.
    completed = subprocess.run([sys.executable, "scripts/run_small_model.py", "--allow-download", "--stage", "description", "--output", str(gpu_out)], cwd=ROOT, env=ENV)
    print({"exit_code": completed.returncode, "output": str(gpu_out), "original_model_equivalent": False})
    if (gpu_out / "error.json").exists():
        print((gpu_out / "error.json").read_text())


After inspecting a valid draft, you may run new separately labeled experiments with `--debloat` or `--stage pipeline`. They use only the synthetic allowlist, reconstructed prompts and the alternative model. Original MCP execution remains disabled, including in Colab. An agent's final text is not a success label; manually review actual tool traces before any task-oriented evaluation.

## Export and recovery

Download results after long stages. If Colab disconnects, re-upload the project to a new runtime, restore only your reviewed checkpoints and use fresh run directories. This notebook does not persist data in Drive. Model caches and package wheels are deliberately excluded from export. No original benchmark is supplied or inferred.


In [ ]:
from google.colab import files
import zipfile
export_path = Path("/content/trustdesc-results.zip")
with zipfile.ZipFile(export_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for base, prefix in [(ROOT / "reports", "reports"), (RUN_ROOT, "current-run")]:
        for p in sorted(base.rglob("*")):
            if p.is_file() and not p.is_symlink():
                z.write(p, str(Path(prefix) / p.relative_to(base)))
files.download(str(export_path))


## Completion criteria

Track A completes only the controlled engineering workflow. Track B produces new inference evidence or documented failures for a different model. Neither supplies the authors' exact prompts, tool/task set or original quantitative reproduction. Compare paper metrics only after authentic artifact recovery, source/license pinning, isolated MCP execution, authorized original-model setup and human-reviewed matched tasks. See the handover guide for the complete remaining-work checklist.
